In [16]:
import os
import subprocess
from pathlib import Path

In [22]:
import av

def get_keyframe_timestamps_pyav(video_path: str):
    keyframe_times = []

    with av.open(video_path) as container:
        stream = container.streams.video[0]

        # Fast path: inspect packets instead of decoding every frame
        for packet in container.demux(stream):
            if not packet.is_keyframe:
                continue

            # Prefer PTS; fallback to DTS if needed
            ts = packet.pts if packet.pts is not None else packet.dts
            if ts is None:
                continue

            t = round(float(ts * packet.time_base), 2)
            keyframe_times.append(t)

    # Optional cleanup
    keyframe_times = sorted(set(keyframe_times))
    return keyframe_times


input_video = Path.cwd().resolve() / "input/franxx_ep.mkv"
keyframe_timestamps = get_keyframe_timestamps_pyav(str(input_video))

print(f"Found {len(keyframe_timestamps)} keyframes")
print(keyframe_timestamps[:20])

Found 395 keyframes
[0.01, 10.02, 11.98, 21.99, 30.62, 39.51, 41.01, 43.01, 45.43, 50.18, 52.35, 54.35, 57.36, 60.36, 61.61, 65.36, 69.41, 71.91, 74.33, 78.25]


In [25]:
import numpy as np
from bisect import bisect_left

path_to_np = "./franxx_scenes_secs.npy"
actual_ep_ranges = np.load(path_to_np)

range_of_eps = []
keyframes = []
for ep_range in actual_ep_ranges:
    range_of_eps.append(float(ep_range[0]))


print(f"range_of_eps: {range_of_eps}")
print(f"keyframe_timestamps: {keyframe_timestamps}")

def compare_ranges_with_tolerance(detected_scenes, keyframes, tolerance_sec=0.2):
    # keyframes should be sorted for bisect
    kf = sorted(float(x) for x in keyframes)

    total_scenes = len(detected_scenes)
    snapped_scene = 0
    matches = []  # (scene_ts, nearest_kf, abs_diff)

    for ts in map(float, detected_scenes):
        i = bisect_left(kf, ts)

        # candidate neighbors around insertion point
        candidates = []
        if i < len(kf):
            candidates.append(kf[i])
        if i > 0:
            candidates.append(kf[i - 1])

        if not candidates:
            continue

        nearest = min(candidates, key=lambda x: abs(x - ts))
        diff = abs(nearest - ts)

        if diff <= tolerance_sec:
            snapped_scene += 1

        matches.append((ts, nearest, diff))

    snap_ratio = (snapped_scene / total_scenes) if total_scenes else 0.0
    return {
        "total_scenes": total_scenes,
        "snapped_scene": snapped_scene,
        "snap_ratio": snap_ratio,
        "matches": matches,
    }

results = compare_ranges_with_tolerance(range_of_eps, keyframe_timestamps)

print(results)

range_of_eps: [0.0, 4.75, 21.52, 24.52, 29.15, 31.32, 34.99, 39.5, 41.0, 52.34, 54.35, 57.35, 60.35, 61.6, 65.36, 69.4, 71.91, 74.32, 78.24, 90.26, 96.76, 103.1, 107.11, 109.53, 117.45, 121.91, 124.92, 129.17, 132.17, 135.68, 141.64, 142.89, 145.6, 147.77, 150.65, 154.07, 156.53, 163.54, 166.04, 171.05, 172.55, 174.55, 176.34, 182.68, 188.06, 192.48, 194.49, 196.99, 198.95, 203.37, 212.34, 220.35, 225.73, 229.69, 234.19, 236.69, 239.7, 243.2, 248.71, 255.46, 258.68, 261.05, 269.81, 275.82, 285.33, 294.21, 300.22, 305.14, 306.26, 308.81, 310.27, 312.65, 315.15, 317.4, 320.9, 324.91, 329.91, 331.91, 337.42, 340.67, 342.09, 348.06, 357.07, 359.57, 361.32, 364.07, 365.78, 368.41, 369.04, 369.95, 372.96, 375.96, 378.96, 381.71, 385.89, 388.6, 393.68, 399.07, 405.07, 410.33, 412.83, 420.84, 432.22, 443.36, 448.49, 449.82, 450.7, 451.78, 453.24, 456.12, 458.12, 460.59, 463.59, 467.43, 472.43, 474.81, 476.77, 483.27, 490.16, 491.74, 496.25, 501.75, 505.21, 507.17, 510.3, 516.14, 516.97, 519.85

In [ ]:
def nearest_within_threshold(sorted_keyframes, ts, threshold):
    i = bisect_left(sorted_keyframes, ts)
    candidates = []
    if i < len(sorted_keyframes):
        candidates.append(sorted_keyframes[i])
    if i > 0:
        candidates.append(sorted_keyframes[i - 1])
    if not candidates:
        return None, None

    nearest = min(candidates, key=lambda k: abs(k - ts))
    diff = abs(nearest - ts)
    if diff <= threshold:
        return nearest, diff
    return None, diff



def compute_scene_range(scenes_secs, keyframe_timestamps, threshold=0.2):
    if threshold < 0:
        raise ValueError(f"Cannot have negative threshold ({threshold})")

    kf = sorted(float(x) for x in keyframe_timestamps)
    final_scene_cuts = []

    for idx, scene in enumerate(scenes_secs):
        scene_start = float(scene[0])
        scene_end = float(scene[1])

        snapped_start, start_diff = nearest_within_threshold(kf, scene_start, threshold)
        snapped_end, end_diff = nearest_within_threshold(kf, scene_end, threshold)

        start_out = snapped_start if snapped_start is not None else scene_start
        end_out = snapped_end if snapped_end is not None else scene_end

        start_snapped = snapped_start is not None
        end_snapped = snapped_end is not None

        if start_snapped and end_snapped:
            mode = "copy_candidate"
        else:
            mode = "reencode_candidate"

        final_scene_cuts.append({
            "scene_id": idx,
            "orig_start": scene_start,
            "orig_end": scene_end,
            "start": start_out,
            "end": end_out,
            "start_snapped": start_snapped,
            "end_snapped": end_snapped,
            "start_diff_sec": start_diff,
            "end_diff_sec": end_diff,
            "mode": mode
        })

    return final_scene_cuts

print(f"actual_ep_ranges: {actual_ep_ranges}")
print(f"keyframe_timestamps: {keyframe_timestamps}")

results = compute_scene_range(actual_ep_ranges, keyframe_timestamps, threshold=0.2)
results_keyframes = []
results_reencode = []
for i in results:
    if i["mode"] == "reencode_candidate":
        results_keyframes.append(i)
    else:
        results_reencode.append(i)


actual_ep_ranges: [[   0.      4.71]
 [   4.75   21.48]
 [  21.52   24.48]
 [  24.52   29.11]
 [  29.15   31.28]
 [  31.32   34.95]
 [  34.99   39.46]
 [  39.5    40.96]
 [  41.     52.3 ]
 [  52.34   54.3 ]
 [  54.35   57.31]
 [  57.35   60.31]
 [  60.35   61.56]
 [  61.6    65.32]
 [  65.36   69.36]
 [  69.4    71.86]
 [  71.91   74.28]
 [  74.32   78.2 ]
 [  78.24   90.22]
 [  90.26   96.72]
 [  96.76  103.06]
 [ 103.1   107.07]
 [ 107.11  109.48]
 [ 109.53  117.41]
 [ 117.45  121.87]
 [ 121.91  124.87]
 [ 124.92  129.13]
 [ 129.17  132.13]
 [ 132.17  135.64]
 [ 135.68  141.6 ]
 [ 141.64  142.85]
 [ 142.89  145.56]
 [ 145.6   147.73]
 [ 147.77  150.61]
 [ 150.65  154.03]
 [ 154.07  156.49]
 [ 156.53  163.5 ]
 [ 163.54  166.  ]
 [ 166.04  171.  ]
 [ 171.05  172.51]
 [ 172.55  174.51]
 [ 174.55  176.3 ]
 [ 176.34  182.64]
 [ 182.68  188.02]
 [ 188.06  192.44]
 [ 192.48  194.44]
 [ 194.49  196.95]
 [ 196.99  198.91]
 [ 198.95  203.33]
 [ 203.37  212.3 ]
 [ 212.34  220.3 ]
 [ 220.35  22

## Function to split final video

Now that we have the functions to detect the keyframes, and whether they should snap to the nearest, and have them as seperated lists, we create a function to first split the keyframe scenes, then to split the reencoded scenes.

In [31]:
def split_final_video(input_file, reencode_scenes, keyframe_scenes, output_dir):
    manifest = {
        "copy_outputs": [],
        "reencode_outputs": [],
    }

    def run_ffmpeg_checked(cmd):
        p = subprocess.run(cmd, capture_output=True, text=True)
        if p.returncode != 0:
            print("FFMPEG CMD:", " ".join(map(str, cmd)))
            print("FFMPEG STDERR:\n", p.stderr)
            raise RuntimeError("ffmpeg failed")
        return p
    
    print("Splitting keyframe copy candidates...")
    for scene in keyframe_scenes:
        scene_id = int(scene["scene_id"])
        start_sec = float(scene["start"])
        end_sec = float(scene["end"])

        if end_sec <= start_sec:
            print(f"Skipping scene {scene_id}: invalid range {start_sec} -> {end_sec}")
            continue

        out_path = output_dir / f"scene_{scene_id:04d}_copy.mp4"
        duration = end_sec - start_sec

        cmd_copy = [
            "ffmpeg",
            "-y",
            "-ss", str(start_sec),
            "-i", str(input_file),
            "-t", str(duration),
            "-map", "0:v:0",
            "-an",
            "-c:v", "copy",
            "-movflags", "+faststart",
            str(out_path),
        ]

        cmd_results = run_ffmpeg_checked(cmd_copy)
        manifest["copy_outputs"].append({
            "scene_id": scene_id,
            "path": str(out_path),
            "mode": "copy",
        })

    print("Splitting and re-encoding non-keyframe candidates...")
    for scene in reencode_scenes:
        scene_id = int(scene["scene_id"])
        start_sec = float(scene["orig_start"])
        end_sec = float(scene["orig_end"])

        if end_sec <= start_sec:
            print(f"Skipping scene {scene_id}: invalid range {start_sec} -> {end_sec}")
            continue

        out_path = output_dir / f"scene_{scene_id:04d}_reencode.mp4"

        cmd_reencode = [
            "ffmpeg",
            "-y",
            "-ss", str(start_sec),
            "-to", str(end_sec),
            "-i", str(input_file),
            "-map", "0:v:0",
            "-an",
            "-c:v", "libx264",
            "-preset", "slow",
            "-crf", "16",
            "-pix_fmt", "yuv420p",
            str(out_path),
        ]

        cmd_results = run_ffmpeg_checked(cmd_reencode)
        manifest["reencode_outputs"].append({
            "scene_id": scene_id,
            "path": str(out_path),
            "mode": "reencode",
        })

    print("Done splitting scenes.")
    print(f"Copy scenes: {len(manifest['copy_outputs'])}")
    print(f"Re-encoded scenes: {len(manifest['reencode_outputs'])}")
    return manifest

output = Path.cwd().resolve() / "output"
result = split_final_video(input_video, reencode_scenes=results_reencode, keyframe_scenes=results_keyframes, output_dir=output)

Splitting keyframe copy candidates...
Splitting and re-encoding non-keyframe candidates...


KeyboardInterrupt: 

### Potential final code block